# Generate tool-use SFT traces (v2 — hybrid)

Three-layer pipeline:
1. **Layer 1 — Programmatic** (Python, no API): calculator templates × random numbers + time templates. Guaranteed unique, valid expressions.
2. **Layer 2 — Multi-model LLM** (OpenRouter): topic-seeded generation with model rotation (DeepSeek V3.1 + GPT-5.4). No tool execution — Result tokens are masked from loss, so LLM-proposed results are fine.
3. **Layer 3 — Dedup + quality filter**: Python exact/semantic dedup + LLM quality scoring. Drops the bottom ~10–15%.

**Before running:**
- Set `OPENROUTER_API_KEY` in Colab Secrets
- CPU runtime is fine (no model loading)

Estimated cost: ~$3–4 via OpenRouter. Time: ~15–20 min.

## 1. Setup

In [ ]:
!pip install -q openai

In [ ]:
import os, json, time, random, math, ast
from collections import Counter
from openai import OpenAI
from google.colab import drive, userdata
from tqdm import tqdm

drive.mount('/content/drive')

SPARKYLLM = '/content/drive/MyDrive/sparkyllm'
OUT_DIR = os.path.join(SPARKYLLM, 'datasets_sft', 'tool_traces')
os.makedirs(OUT_DIR, exist_ok=True)

GROUNDED_FILE = os.path.join(OUT_DIR, 'tool_traces_grounded.jsonl')
META_FILE     = os.path.join(OUT_DIR, 'generate_meta.json')

# OpenRouter client
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
assert OPENROUTER_API_KEY, 'Set OPENROUTER_API_KEY in Colab Secrets first'
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url='https://openrouter.ai/api/v1')
print('OpenRouter client ready')

## 2. Models + topic seeds

In [ ]:
# Model roster — rotated across batches for natural diversity
MODELS = [
    'deepseek/deepseek-chat-v3.1',
    'openai/gpt-5.4',
]

BATCH_SIZE = 10  # questions per API call
GEN_TEMPERATURE = 0.9
QUALITY_TEMPERATURE = 0.1

# ---- Topic seeds (one list per LLM-generated category) ----

CALC_WORD_TOPICS = [
    'cooking recipes', 'grocery shopping', 'road trip distances', 'sports statistics',
    'home renovation', 'personal finance', 'restaurant tipping', 'unit conversion',
    'recipe scaling', 'fuel economy', 'salary calculation', 'time zones',
    'temperature conversion', 'population statistics', 'farming yields',
    'manufacturing costs', 'shipping rates', 'electricity bills', 'paint coverage',
    'carpet area', 'vehicle speed', 'weight conversion', 'chained discounts',
    'compound interest', 'classroom supplies',
]

WEB_SEARCH_TOPICS = [
    'ancient civilizations', 'marine biology', 'space exploration', 'climate change',
    'classical music', 'world capitals', 'human nutrition', 'famous architecture',
    'film history', 'quantum physics', 'renaissance art', 'world rivers',
    'endangered species', 'volcanic eruptions', 'ancient mythology', 'Nobel Prize winners',
    'Olympic records', 'programming languages', 'medieval history', 'deep sea creatures',
    'mountain peaks', 'deserts of the world', 'famous inventors', 'infectious diseases',
    'renewable energy', 'the solar system', 'famous philosophers', 'world religions',
    'national parks', 'bridges and tunnels', 'historical treaties', 'famous scientists',
    'literary classics', 'musical instruments', 'martial arts', 'ancient languages',
    'tropical rainforests', 'Arctic exploration', 'famous battles', 'international cuisine',
    'photography history', 'dog breeds', 'gemstones and minerals', 'world currencies',
    'space telescopes', 'ancient Egypt', 'modern dance', 'nuclear energy',
    'human rights history', 'board games history', 'the periodic table', 'wonders of the world',
    'famous speeches', 'coral reefs', 'weather phenomena', 'constellations',
    'history of aviation', 'world trade routes', 'artificial intelligence history',
    'traditional medicine', 'ocean currents', 'famous libraries', 'civil engineering marvels',
    'cartography', 'bird migration', 'ancient Rome', 'famous gardens',
    'fossils and paleontology', 'cryptography history', 'history of vaccines',
    'native American history', 'silk road history', 'famous shipwrecks',
    'evolution of computers', 'history of chocolate', 'famous caves',
]

NONE_TOPICS = [
    'greetings', 'farewells', 'gratitude', 'compliments', 'opinions on food',
    'opinions on music', 'opinions on movies', 'opinions on books', 'travel preferences',
    'sports opinions', 'weather small talk', 'technology opinions', 'fashion thoughts',
    'career advice', 'health tips', 'relationship advice', 'financial wisdom',
    'education thoughts', 'cooking tips', 'fitness motivation', 'jokes',
    'riddles', 'fun facts', 'motivational quotes', 'philosophical questions',
    'hypothetical scenarios', 'favorite colors', 'favorite seasons', 'favorite animals',
    'hobby recommendations', 'drink preferences', 'city comparisons', 'weekend plans',
    'family conversation', 'workplace chat', 'food recommendations', 'creative stories',
    'short poems', 'song ideas', 'self reflection', 'emotions and feelings',
    'childhood memories', 'dream interpretation', 'goal setting', 'overcoming challenges',
    'celebrating achievements', 'learning new skills', 'expressing curiosity',
    'expressing confusion', 'agreeing with someone', 'polite disagreement',
    'expressing surprise', 'showing disappointment', 'sharing excitement',
    'offering sympathy', 'giving encouragement', 'expressing optimism',
    'nostalgic conversation', 'anticipation', 'deep gratitude', 'sincere apology',
    'celebration', 'humor and sarcasm', 'formal conversation', 'casual banter',
    'enthusiasm about a topic', 'healthy skepticism', 'brainstorming ideas',
    'recommending restaurants', 'discussing sleep habits', 'pet stories',
    'board game chat', 'gardening chat', 'road trip stories', 'concert experiences',
    'museum visits', 'volunteering', 'book club discussion', 'movie reviews',
]

MULTI_STEP_TYPES = [
    ('calc_then_compare', 50, 'Step 1 uses calculator, step 2 uses none (compare the result against a threshold). Example: "What is 17*23, and is that bigger than 400?"'),
    ('calc_then_calc', 100, 'Both steps use calculator. Example: "What is 15+27, and what is that divided by 3?"'),
    ('search_then_reason', 100, 'Step 1 uses web_search, step 2 uses none (reason about what was found). Example: "Look up the population of Tokyo. Is it more than 10 million?"'),
    ('time_then_calc', 100, 'Step 1 uses time, step 2 uses calculator. Example: "What time is it now, and what time will it be in 3.5 hours?"'),
    ('search_then_search', 100, 'Both steps use web_search. Example: "Who wrote The Odyssey and when were they born?"'),
    ('calc_then_time', 100, 'Step 1 uses calculator, step 2 uses time. Example: "What is 15*4? Also, what time is it now?"'),
    ('calc_then_search', 50, 'Step 1 uses calculator, step 2 uses web_search. Example: "What is 12*12? Also, who invented the lightbulb?"'),
    ('search_then_calc', 50, 'Step 1 uses web_search, step 2 uses calculator. Example: "How tall is the Eiffel Tower in feet? Convert that to meters by multiplying by 0.3048."'),
    ('time_then_search', 50, 'Step 1 uses time, step 2 uses web_search. Example: "What time is it? And who was the first person on the moon?"'),
]

print(f'Calculator word topics: {len(CALC_WORD_TOPICS)}')
print(f'Web search topics: {len(WEB_SEARCH_TOPICS)}')
print(f'None topics: {len(NONE_TOPICS)}')
print(f'Multi-step types: {len(MULTI_STEP_TYPES)} ({sum(c for _, c, _ in MULTI_STEP_TYPES)} traces)')

## 3. Layer 1 — Programmatic traces (calculator + time)

No API calls. Calculator traces from templates × random numbers. Time traces from phrasing variants. All traces have real computed results.

In [ ]:
random.seed(42)

# ---- Calculator templates ----
# Each: (question_fmt, thought, expr_fmt, final_fmt, uses_pct)
# uses_pct=True means the template expects {pct_dec} — allowed to have decimals.
CALC_TMPLS = [
    ('What is {a} plus {b}?', 'I need to add these numbers.', '{a} + {b}', '{a} plus {b} is {result}.', False),
    ('What is {a} minus {b}?', 'I need to subtract.', '{a} - {b}', '{a} minus {b} is {result}.', False),
    ('What is {a} times {b}?', 'I need to multiply.', '{a} * {b}', '{a} times {b} is {result}.', False),
    ('What is {a} divided by {b}?', 'I need to divide.', '{a} / {b}', '{a} divided by {b} is {result}.', False),
    ('What is {pct}% of {b}?', 'I need to calculate a percentage.', '{pct_dec} * {b}', '{pct}% of {b} is {result}.', True),
    ('If I buy {n} items at ${price} each, what is the total?', 'I need to multiply quantity by price.', '{n} * {price}', 'The total for {n} items at ${price} each is ${result}.', False),
    ('What is the area of a rectangle {a} by {b}?', 'Area is length times width.', '{a} * {b}', 'The area is {result} square units.', False),
    ('What is {a} squared?', 'I need to square the number.', '{a} ** 2', '{a} squared is {result}.', False),
    ('What is the square root of {a}?', 'I need the square root.', '{a} ** 0.5', 'The square root of {a} is {result}.', False),
    ('How much is {a} to the power of {n}?', 'I need to compute an exponent.', '{a} ** {n}', '{a} to the power of {n} is {result}.', False),
    ('What is the remainder when {a} is divided by {b}?', 'I will use the modulo operator.', '{a} % {b}', 'The remainder is {result}.', False),
    ('What is {pct}% tip on a ${price} bill?', 'I need to calculate the tip.', '{pct_dec} * {price}', 'The tip is ${result}.', True),
    ('If something costs ${price} and is {pct}% off, what is the discount amount?', 'I need to compute the discount.', '{pct_dec} * {price}', 'The discount is ${result}.', True),
    ('What is the average of {a}, {b}, and {c}?', 'I need to sum and divide by 3.', '({a} + {b} + {c}) / 3', 'The average is {result}.', False),
    ('If I drive {a} miles in {b} hours, what is my speed?', 'Speed is distance divided by time.', '{a} / {b}', 'The speed is {result} mph.', False),
    ('Convert {a} feet to meters.', 'I multiply by 0.3048.', '{a} * 0.3048', '{a} feet is {result} meters.', False),
    ('Convert {a} kilograms to pounds.', 'I multiply by 2.20462.', '{a} * 2.20462', '{a} kg is {result} pounds.', False),
    ('What is {a} plus {pct}% of {a}?', 'I need to add the percentage to the original.', '{a} + {pct_dec} * {a}', 'The result is {result}.', True),
    ('If I split ${price} equally among {n} people, how much does each pay?', 'I divide the total by the number of people.', '{price} / {n}', 'Each person pays ${result}.', False),
    ('What is {a} times {b} minus {c}?', 'I need to multiply then subtract.', '{a} * {b} - {c}', 'The result is {result}.', False),
]

def make_calc_trace():
    tmpl = random.choice(CALC_TMPLS)
    q_fmt, thought, expr_fmt, final_fmt, uses_pct = tmpl
    # Force integer operands for non-percentage templates to avoid
    # teaching the model that decimals are normal in arithmetic.
    if uses_pct:
        a = random.choice([random.randint(2, 500), round(random.uniform(1, 100), 1)])
        b = random.choice([random.randint(2, 500), round(random.uniform(1, 100), 1)])
    else:
        a = random.randint(2, 500)
        b = random.randint(2, 500)
    c = random.randint(2, 200)
    n = random.randint(2, 20)
    pct = random.choice([5, 10, 12, 15, 18, 20, 25, 30, 33, 40, 50, 75])
    pct_dec = pct / 100
    price = round(random.uniform(5, 500), 2)
    vals = dict(a=a, b=b, c=c, n=n, pct=pct, pct_dec=pct_dec, price=price)
    try:
        q = q_fmt.format(**vals)
        expr = expr_fmt.format(**vals)
        result = eval(expr)
        if isinstance(result, float):
            result = round(result, 4)
            if result == int(result): result = int(result)
        final = final_fmt.format(result=result, **vals)
        return {
            'question': q, 'category': 'single_calculator',
            'steps': [{'thought': thought, 'action': 'calculator', 'input': expr, 'result': str(result)}],
            'final': final,
        }
    except Exception:
        return None

# ---- Time templates (varied timestamps) ----
TIME_QUESTIONS = [
    'What time is it?', 'What is the current time?', 'Can you tell me the time?',
    'What is today\'s date?', 'What day of the week is it?', 'Tell me the date and time.',
    'What is the current date?', 'How late is it right now?', 'Do you know what time it is?',
    'What month are we in?', 'What year is it?', 'Tell me today\'s date please.',
    'I need to know the current time.', 'What day is today?', 'Is it morning or afternoon?',
]
TIME_THOUGHTS = [
    'I will check the current time.', 'Let me look up the time.',
    'I need to use the time tool.', 'I will get the current date and time.',
]
TIME_RESULTS = [
    '2026-04-12 15:30:00', '2026-04-12 09:15:42', '2026-04-13 22:05:11',
    '2026-04-10 07:00:33', '2026-04-11 12:45:59', '2026-04-14 18:20:07',
    '2026-04-09 03:55:28', '2026-04-15 16:10:44', '2026-04-08 11:30:16',
    '2026-04-07 20:40:52', '2026-04-06 14:25:38', '2026-04-05 08:50:03',
    '2026-03-28 23:15:19', '2026-03-30 06:35:47', '2026-04-01 17:00:00',
    '2026-04-02 10:22:31', '2026-04-03 13:44:09', '2026-04-04 19:58:55',
    '2026-03-25 04:12:26', '2026-03-27 21:33:40',
]
TIME_FINALS = [
    'The current time is {r}.', 'It is {r} right now.', 'Right now it is {r}.',
    'The date and time is {r}.', 'As of now, the time is {r}.',
]

def make_time_trace(q_idx):
    q = TIME_QUESTIONS[q_idx % len(TIME_QUESTIONS)]
    r = TIME_RESULTS[q_idx % len(TIME_RESULTS)]
    return {
        'question': q, 'category': 'single_time',
        'steps': [{'thought': random.choice(TIME_THOUGHTS), 'action': 'time', 'input': '', 'result': r}],
        'final': random.choice(TIME_FINALS).format(r=r),
    }

# ---- Generate Layer 1 ----
layer1_traces = []
seen_questions = set()

# Calculator: 750 unique traces (up from 500 — guaranteed-correct programmatic traces)
attempts = 0
while len([t for t in layer1_traces if t['category'] == 'single_calculator']) < 750 and attempts < 3000:
    t = make_calc_trace()
    attempts += 1
    if t and t['question'] not in seen_questions:
        seen_questions.add(t['question'])
        layer1_traces.append(t)

# Time: 150 traces (cycle through templates with varied timestamps)
for i in range(150):
    t = make_time_trace(i)
    if t['question'] not in seen_questions:
        seen_questions.add(t['question'])
        layer1_traces.append(t)

cat_counts = Counter(t['category'] for t in layer1_traces)
print(f'Layer 1: {len(layer1_traces)} traces')
for cat, n in cat_counts.most_common():
    print(f'  {cat}: {n}')

## 4. Layer 2 — LLM generation with model rotation + topic seeds

No tool execution. The LLM proposes ALL fields including Result (which is masked from loss anyway). Model rotation between DeepSeek V3.1 and GPT-5.4 for diversity.

In [ ]:
SINGLE_TOOL_SYSTEM = """You are generating training data for a small language model that will learn to use tools.

You will generate questions in a specific category. Return ONLY valid JSON with this structure:
{
  "traces": [
    {
      "question": "the user's question",
      "thought": "one sentence explaining what to do",
      "action": "tool name",
      "input": "tool input",
      "result": "plausible tool output",
      "final": "one or two sentence answer to the user"
    }
  ]
}

Rules:
- For action=calculator: input MUST be a valid Python arithmetic expression (only numbers, +, -, *, /, //, %, **, parentheses). NO variable names, NO words. Example: 0.15 * 240
- For action=web_search: input is a 3-8 word search query. result is a 1-3 sentence factual summary.
- For action=time: input is empty string. result is a timestamp like 2026-04-12 15:30:00.
- For action=none: input is empty string. result is empty string. The model answers the question directly.
- Each question must be UNIQUE and different from others in this batch.
- thought must be ONE short sentence.
- final must be ONE or TWO sentences that answer the question using the result.
- The final answer MUST directly use the value from result. Do not paraphrase, round, or add information not present in the result."""

MULTI_STEP_SYSTEM = """You are generating training data for a small language model that will learn to use tools in multiple steps.

Return ONLY valid JSON with this structure:
{
  "traces": [
    {
      "question": "the user's question",
      "steps": [
        {"thought": "...", "action": "...", "input": "...", "result": "..."},
        {"thought": "...", "action": "...", "input": "...", "result": "..."}
      ],
      "final": "one or two sentence answer"
    }
  ]
}

Each trace must have EXACTLY 2 steps. Tool options: calculator, web_search, time, none.
For calculator: input MUST be a valid Python arithmetic expression (numbers and operators only, NO words).
For web_search: input is a short search query, result is a factual summary.
For time: input is empty string, result is a timestamp.
For none: input and result are both empty strings.
Each question must be UNIQUE. Thoughts must be one short sentence.
The final answer MUST directly incorporate the values from both step results. Do not paraphrase, round, or add information not present in the results."""

In [ ]:
def llm_generate(system_prompt, user_msg, model):
    for attempt in range(3):
        try:
            comp = client.chat.completions.create(
                model=model,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user', 'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                temperature=GEN_TEMPERATURE,
                max_tokens=4000,
            )
            raw = comp.choices[0].message.content
            data = json.loads(raw)
            return data.get('traces', [])
        except Exception as e:
            if attempt == 2:
                print(f'  FAILED ({model}): {e}')
                return []
            time.sleep(2)
    return []

def normalize_single(raw_trace):
    """Convert a single-tool raw trace dict to our standard format."""
    if not isinstance(raw_trace, dict): return None
    q = raw_trace.get('question', '').strip()
    if not q: return None
    action = (raw_trace.get('action') or '').lower().strip()
    return {
        'question': q,
        'category': f'single_{action}' if action in ('calculator', 'web_search', 'time', 'none') else None,
        'steps': [{
            'thought': (raw_trace.get('thought') or '').strip(),
            'action': action,
            'input': str(raw_trace.get('input', '')).strip(),
            'result': str(raw_trace.get('result', '')).strip(),
        }],
        'final': (raw_trace.get('final') or '').strip(),
    }

def normalize_multi(raw_trace):
    """Convert a multi-step raw trace dict to our standard format."""
    if not isinstance(raw_trace, dict): return None
    q = raw_trace.get('question', '').strip()
    steps_raw = raw_trace.get('steps', [])
    if not q or not isinstance(steps_raw, list) or len(steps_raw) < 2: return None
    steps = []
    for s in steps_raw[:2]:
        if not isinstance(s, dict): return None
        action = (s.get('action') or '').lower().strip()
        if action not in ('calculator', 'web_search', 'time', 'none'): return None
        steps.append({
            'thought': (s.get('thought') or '').strip(),
            'action': action,
            'input': str(s.get('input', '')).strip(),
            'result': str(s.get('result', '')).strip(),
        })
    return {
        'question': q, 'category': 'multi_step', 'steps': steps,
        'final': (raw_trace.get('final') or '').strip(),
    }

# ---- Generate all Layer 2 categories ----
layer2_traces = []
call_count = 0

def gen_single_category(category, action, topics, target):
    global call_count
    traces = []
    for i, topic in enumerate(topics):
        if len(traces) >= target: break
        model = MODELS[i % len(MODELS)]
        user_msg = f'Generate {BATCH_SIZE} diverse questions about "{topic}" where the best tool is {action}. Each question must be unique and different.'
        if action == 'calculator':
            user_msg += ' Remember: calculator input must be a VALID Python arithmetic expression with numbers only, no variable names.'
        raw = llm_generate(SINGLE_TOOL_SYSTEM, user_msg, model)
        call_count += 1
        for r in raw:
            t = normalize_single(r)
            if t and t['category'] == f'single_{action}' and t['question'] not in seen_questions:
                # Override the action in case the LLM picked a different one
                t['steps'][0]['action'] = action
                t['category'] = f'single_{action}'
                seen_questions.add(t['question'])
                traces.append(t)
    return traces[:target]

print('Generating calculator word problems...')
layer2_traces += gen_single_category('single_calculator', 'calculator', CALC_WORD_TOPICS, 250)
print(f'  got {len([t for t in layer2_traces if t["category"]=="single_calculator"])}')

print('Generating web_search traces...')
layer2_traces += gen_single_category('single_web_search', 'web_search', WEB_SEARCH_TOPICS, 900)
print(f'  got {len([t for t in layer2_traces if t["category"]=="single_web_search"])}')

print('Generating none (direct answer) traces...')
layer2_traces += gen_single_category('single_none', 'none', NONE_TOPICS, 750)
print(f'  got {len([t for t in layer2_traces if t["category"]=="single_none"])}')

# Multi-step
print('Generating multi-step traces...')
for subcat, target, extra_instruction in MULTI_STEP_TYPES:
    got = 0
    batches_needed = (target + BATCH_SIZE - 1) // BATCH_SIZE
    for batch_i in range(batches_needed):
        if got >= target: break
        model = MODELS[batch_i % len(MODELS)]
        user_msg = f'Generate {BATCH_SIZE} diverse multi-step questions. {extra_instruction} Each question must be unique.'
        raw = llm_generate(MULTI_STEP_SYSTEM, user_msg, model)
        call_count += 1
        for r in raw:
            t = normalize_multi(r)
            if t and t['question'] not in seen_questions:
                seen_questions.add(t['question'])
                layer2_traces.append(t)
                got += 1
    print(f'  {subcat}: {got}')

print(f'\nLayer 2 complete: {len(layer2_traces)} traces in {call_count} API calls')
cat2 = Counter(t['category'] for t in layer2_traces)
for cat, n in cat2.most_common():
    print(f'  {cat}: {n}')

## 5. Layer 3 — Dedup + validate + LLM quality score

1. Python: exact dedup, validate calculator expressions with `ast.parse`, drop empty/malformed fields.
2. LLM: quality score 1–5 via DeepSeek V3.1. Drop traces scoring below 3.

In [ ]:
import re

all_traces = layer1_traces + layer2_traces
print(f'Pre-filter: {len(all_traces)} total traces')

# ---- Exact dedup ----
deduped = []
qs = set()
dup_count = 0
for t in all_traces:
    q = t['question'].strip().lower()
    if q in qs:
        dup_count += 1
        continue
    qs.add(q)
    deduped.append(t)
print(f'After exact dedup: {len(deduped)} (removed {dup_count} dupes)')

# ---- Field validation ----
_DECIMAL_RE = re.compile(r'\d+\.\d+')

valid = []
bad_calc = 0
bad_decimal = 0
bad_field = 0
for t in deduped:
    ok = True
    for step in t['steps']:
        if step['action'] not in ('calculator', 'web_search', 'time', 'none'):
            ok = False; break
        if not step.get('thought'):
            ok = False; break
        # Validate calculator input parses as Python expression
        if step['action'] == 'calculator':
            try:
                ast.parse(step['input'], mode='eval')
            except SyntaxError:
                bad_calc += 1
                ok = False; break
            # Reject decimal-in-input when the question doesn't mention %/percent
            # (prevents the model from learning to convert integers to decimals)
            if _DECIMAL_RE.search(step['input']):
                q_lower = t['question'].lower()
                if '%' not in q_lower and 'percent' not in q_lower:
                    bad_decimal += 1
                    ok = False; break
    if not t.get('final'):
        ok = False
        bad_field += 1
    if ok:
        valid.append(t)
    elif not ok and bad_calc == 0 and bad_decimal == 0:
        bad_field += 1
print(f'After validation: {len(valid)} (bad calc expr: {bad_calc}, decimal mismatch: {bad_decimal}, bad fields: {bad_field})')

In [ ]:
# ---- LLM quality scoring ----
QUALITY_SYSTEM = 'You are a data quality reviewer. You will see tool-use traces for training an AI model. Score each trace 1-5 on overall quality (correct tool choice, sensible input, coherent final answer). Return JSON: {"scores": [3, 5, 4, ...]}'

def score_batch(batch):
    text_parts = []
    for i, t in enumerate(batch, 1):
        steps_text = ''
        for s in t['steps']:
            steps_text += f'  Thought: {s["thought"]}\n  Action: {s["action"]}\n  Input: {s["input"]}\n  Result: {s["result"]}\n'
        text_parts.append(f'Trace {i}:\n  Question: {t["question"]}\n{steps_text}  Final: {t["final"]}')
    user_msg = 'Score each trace 1-5 (1=bad, 5=perfect). Return JSON {"scores": [...]}\n\n' + '\n\n'.join(text_parts)
    for attempt in range(3):
        try:
            comp = client.chat.completions.create(
                model=MODELS[0],  # cheapest model for scoring
                messages=[
                    {'role': 'system', 'content': QUALITY_SYSTEM},
                    {'role': 'user', 'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                temperature=QUALITY_TEMPERATURE,
                max_tokens=500,
            )
            data = json.loads(comp.choices[0].message.content)
            scores = data.get('scores', [])
            if isinstance(scores, list) and len(scores) == len(batch):
                return [int(s) if isinstance(s, (int, float)) else 3 for s in scores]
        except Exception:
            if attempt < 2: time.sleep(2)
    return [3] * len(batch)  # default to passing score on failure

QUALITY_BATCH = 10
MIN_SCORE = 3
scored = []
dropped_quality = 0

print(f'Quality scoring {len(valid)} traces...')
for i in tqdm(range(0, len(valid), QUALITY_BATCH), desc='scoring'):
    batch = valid[i:i+QUALITY_BATCH]
    scores = score_batch(batch)
    for trace, score in zip(batch, scores):
        if score >= MIN_SCORE:
            trace['quality_score'] = score
            scored.append(trace)
        else:
            dropped_quality += 1

print(f'After quality filter: {len(scored)} (dropped {dropped_quality} below score {MIN_SCORE})')

cat_final = Counter(t['category'] for t in scored)
print('\nFinal distribution:')
for cat, n in cat_final.most_common():
    print(f'  {cat:25s} {n}')

## 6. Save grounded traces + metadata

In [ ]:
# Write final JSONL
with open(GROUNDED_FILE, 'w') as f:
    for t in scored:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')
print(f'Wrote {len(scored)} traces to {GROUNDED_FILE}')

# Metadata
meta = {
    'source': 'hybrid programmatic + multi-model LLM (OpenRouter)',
    'models': MODELS,
    'total_api_calls': call_count,
    'layer1_count': len(layer1_traces),
    'layer2_count': len(layer2_traces),
    'pre_filter_total': len(all_traces),
    'duplicates_removed': dup_count,
    'bad_calc_expressions': bad_calc,
    'bad_fields': bad_field,
    'dropped_quality': dropped_quality,
    'final_count': len(scored),
    'final_distribution': dict(cat_final),
}
with open(META_FILE, 'w') as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))
print()
print('Next: run tokenize_tool_sft.ipynb, then sft_tool_use.ipynb')